# 03 - Guardrails & Checks
**Module:** `src/guardrails/checks.py`

## What this module does?
Runs validation checks at two moments in the pipeline:

- **Pre-generation** — validates the inputs before any API call is made
- **Post-generation** — validates the lesson the LLM returned

Every check returns a `CheckResult` with a status (`pass` or `flag`)
and a human-readable message. Flags never crash the system, they
are recorded inside the lesson JSON for transparency.

## Research grounding
- **Cognitive load checks** — Sweller (1988): grade-band rules on working memory
- **Single skill rule** — Rosenshine (2012): one clear objective per lesson
- **Cultural bias scan** — Brief requirement: known limitation of narrative AI generation
- **Vocabulary ceiling** — Chall (1983): developmental vocabulary expectations

## What we'll explore?
1. Pre-generation checks on valid and invalid inputs
2. The deduplication check
3. Post-generation checks on lesson content
4. Running all post-checks at once and reading the output

In [1]:
import sys
sys.path.insert(0, '..')

from src.guardrails.checks import (
    CheckResult,
    validate_grade_band,
    validate_ela_domain,
    validate_theme,
    check_deduplication,
    check_single_skill,
    check_vocabulary_ceiling,
    check_cognitive_load,
    check_cultural_bias,
    run_pre_checks,
    run_post_checks,
)


## Part 1: The CheckResult Object
Every check returns a `CheckResult`. Let's understand it before
running any checks.

In [2]:
# Create CheckResults manually to understand the structure
passing = CheckResult(status="pass", message="Everything looks good.")
flagged = CheckResult(status="flag", message="Something needs review.")

print("Passing result:", passing)
print("Flagged result:", flagged)
print()
print("passing.passed() →", passing.passed())
print("flagged.passed() →", flagged.passed())

Passing result: ✅ [PASS] Everything looks good.
Flagged result: ⚠️ [FLAG] Something needs review.

passing.passed() → True
flagged.passed() → False


## Part 2: Pre-Generation Checks
These run before the prompt is built.
A failed pre-check means we never call the API...this is saving tokens and time.


In [3]:
# Valid inputs — all should pass
print("=== VALID INPUTS ===\n")
print(validate_grade_band("6-8"))
print(validate_ela_domain("Speaking"))
print(validate_theme("Climate change"))

=== VALID INPUTS ===

✅ [PASS] Grade band '6-8' is valid.
✅ [PASS] ELA domain 'Speaking' is valid.
✅ [PASS] Theme 'Climate change' is valid.


In [4]:
# Invalid inputs — all should flag
print("=== INVALID INPUTS ===\n")
print(validate_grade_band("K-12"))
print(validate_ela_domain("Listening and Speaking"))
print(validate_theme(""))

=== INVALID INPUTS ===

⚠️ [FLAG] 'K-12' is not a valid grade band. Must be one of: ['3-5', '6-8', '9-12', 'K-2']
⚠️ [FLAG] 'Listening and Speaking' is not a valid ELA domain. Must be one of: ['Listening', 'Reading', 'Reading → Speaking', 'Speaking', 'Writing']
⚠️ [FLAG] Theme cannot be empty.


In [5]:
# Theme edge cases
print("=== THEME EDGE CASES ===\n")
print(validate_theme(""))               # empty
print(validate_theme("  "))            # whitespace only
print(validate_theme("AI"))            # too short
print(validate_theme("A" * 81))        # too long
print(validate_theme("Ocean Ecosystems"))  # valid

=== THEME EDGE CASES ===

⚠️ [FLAG] Theme cannot be empty.
⚠️ [FLAG] Theme cannot be empty.
⚠️ [FLAG] Theme 'AI' is too short. Please provide a descriptive theme.
⚠️ [FLAG] Theme 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' is too short. Please provide a descriptive theme.
✅ [PASS] Theme 'Ocean Ecosystems' is valid.


## Part 3: run_pre_checks() — All at Once
In the generator, all three pre-checks run together.
If any fail, a `ValueError` is raised and generation stops immediately.

In [6]:
# Valid inputs — should pass silently
print("=== ALL PRE-CHECKS PASS ===\n")
run_pre_checks("9-12", "Speaking", "Artificial Intelligence")

=== ALL PRE-CHECKS PASS ===

[checks] All pre-generation checks passed ✅


{'grade_band': ✅ [PASS] Grade band '9-12' is valid.,
 'ela_domain': ✅ [PASS] ELA domain 'Speaking' is valid.,
 'theme': ✅ [PASS] Theme 'Artificial Intelligence' is valid.}

In [10]:
# Invalid input — should raise ValueError with clear message
print("=== PRE-CHECKS FAIL ===\n")
try:
    run_pre_checks("kindergarten", "Math", "")
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== PRE-CHECKS FAIL ===

Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] 'kindergarten' is not a valid grade band. Must be one of: ['3-5', '6-8', '9-12', 'K-2']
  ⚠️ [FLAG] 'Math' is not a valid ELA domain. Must be one of: ['Listening', 'Reading', 'Reading → Speaking', 'Speaking', 'Writing']
  ⚠️ [FLAG] Theme cannot be empty.


## Part 4: Post-Generation Checks
These run on the lesson dict returned by the LLM.
Unlike pre-checks, these never raise rather they flag and record.

Let's build a test lesson dict and run each check individually.

In [11]:
# A clean lesson dict — should pass all checks
clean_lesson = {
    "metadata": {
        "grade_band": "3-5",
        "primary_skill": "Explain a process step by step"
    },
    "lesson_flow": {
        "hook": {"content": "A scientist discovers something amazing in the deep ocean."},
        "practice": [
            {"prompt_id": "P1", "text": "Tell me how the creature survives.", "scaffold": "Start with: First..."},
            {"prompt_id": "P2", "text": "Now explain the whole process in your own words.", "scaffold": None}
        ]
    }
}

print("=== SINGLE SKILL CHECK ===")
print(check_single_skill(clean_lesson))

print("\n=== VOCABULARY CEILING CHECK ===")
print(check_vocabulary_ceiling(clean_lesson))

print("\n=== COGNITIVE LOAD CHECK ===")
print(check_cognitive_load(clean_lesson))

print("\n=== CULTURAL BIAS CHECK ===")
print(check_cultural_bias(clean_lesson))

=== SINGLE SKILL CHECK ===
✅ [PASS] Single skill confirmed: 'Explain a process step by step'

=== VOCABULARY CEILING CHECK ===
✅ [PASS] All prompts within 3-5 vocabulary ceiling (60 words).

=== COGNITIVE LOAD CHECK ===
✅ [PASS] Cognitive load check passed for grade band '3-5'.

=== CULTURAL BIAS CHECK ===
✅ [PASS] No cultural bias terms detected.


## Part 5: Triggering Each Flag Deliberately
Now let's craft lessons that deliberately trigger each guardrail.
This confirms the checks are working as designed.

In [12]:
# Trigger: multi-skill
print("=== TRIGGER: Multi-skill ===")
multi_skill = {
    "metadata": {"grade_band": "3-5", "primary_skill": "Retell a story and identify the main idea"},
    "lesson_flow": {"hook": {"content": "Short hook."}, "practice": []}
}
print(check_single_skill(multi_skill))

=== TRIGGER: Multi-skill ===
⚠️ [FLAG] primary_skill may describe more than one skill (contains 'and'): 'Retell a story and identify the main idea'


In [13]:
# Trigger: vocabulary ceiling exceeded for K-2
print("=== TRIGGER: Vocabulary ceiling (K-2) ===")
long_prompt_lesson = {
    "metadata": {"grade_band": "K-2"},
    "lesson_flow": {
        "hook": {"content": "Short."},
        "practice": [
            {"prompt_id": "P1", "text": " ".join(["word"] * 35)}  # 35 words — over K-2 ceiling of 30
        ]
    }
}
print(check_vocabulary_ceiling(long_prompt_lesson))

=== TRIGGER: Vocabulary ceiling (K-2) ===
⚠️ [FLAG] Vocabulary ceiling exceeded: Prompt P1 has 35 words (ceiling for K-2 is 30)


In [14]:
# Trigger: cognitive load — K-2 hook too long
print("=== TRIGGER: Cognitive load (K-2 long hook) ===")
long_hook_lesson = {
    "metadata": {"grade_band": "K-2"},
    "lesson_flow": {
        "hook": {"content": " ".join(["word"] * 90)},  # 90 words — over 80 word limit
        "practice": []
    }
}
print(check_cognitive_load(long_hook_lesson))

=== TRIGGER: Cognitive load (K-2 long hook) ===
⚠️ [FLAG] K-2 hook is 90 words — may introduce too much novelty at once. Consider simplifying to under 80 words (Sweller, 1988).


In [15]:
# Trigger: cultural bias
print("=== TRIGGER: Cultural bias ===")
biased_lesson = {
    "metadata": {"grade_band": "3-5"},
    "lesson_flow": {
        "hook": {"content": "Every Thanksgiving, American families come together to celebrate."},
        "practice": []
    }
}
print(check_cultural_bias(biased_lesson))

=== TRIGGER: Cultural bias ===
⚠️ [FLAG] Potential cultural specificity detected: ['thanksgiving', 'american']. Review whether non-US/non-Western students would have sufficient context. Consider adding a brief explainer or choosing a more globally neutral reference.


## Part 6: run_post_checks() — All at Once
This is what the validator calls after every LLM response.
It runs all four checks and returns a dict of results.



In [17]:
# Run all post checks on the clean lesson
print("=== POST-CHECKS ON CLEAN LESSON ===\n")
results = run_post_checks(clean_lesson)
print()
flags = [name for name, r in results.items() if not r.passed()]
print(f"Flags raised: {len(flags)}")

=== POST-CHECKS ON CLEAN LESSON ===

[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Explain a process step by step'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within 3-5 vocabulary ceiling (60 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band '3-5'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.

Flags raised: 0


In [18]:
# Run all post checks on a problematic lesson
print("=== POST-CHECKS ON PROBLEMATIC LESSON ===\n")
problematic = {
    "metadata": {
        "grade_band": "K-2",
        "primary_skill": "Retell a story and identify the theme"
    },
    "lesson_flow": {
        "hook": {"content": "Every Thanksgiving, American families " + " ".join(["word"] * 70)},
        "practice": [
            {"prompt_id": "P1", "text": " ".join(["word"] * 35)}
        ]
    }
}
results = run_post_checks(problematic)
print()
flags = [name for name, r in results.items() if not r.passed()]
print(f"Flags raised: {len(flags)} — {flags}")

=== POST-CHECKS ON PROBLEMATIC LESSON ===

[checks] single_skill_check: ⚠️ [FLAG] primary_skill may describe more than one skill (contains 'and'): 'Retell a story and identify the theme'
[checks] vocabulary_ceiling_check: ⚠️ [FLAG] Vocabulary ceiling exceeded: Prompt P1 has 35 words (ceiling for K-2 is 30)
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ⚠️ [FLAG] Potential cultural specificity detected: ['thanksgiving', 'american']. Review whether non-US/non-Western students would have sufficient context. Consider adding a brief explainer or choosing a more globally neutral reference.

Flags raised: 3 — ['single_skill_check', 'vocabulary_ceiling_check', 'cultural_bias_check']


## Summary

| Check | Triggers on | Never raises? |
|---|---|---|
| `validate_grade_band` | Invalid grade band string | Pre-check — raises ValueError |
| `validate_ela_domain` | Invalid domain string | Pre-check — raises ValueError |
| `validate_theme` | Empty, too short, too long | Pre-check — raises ValueError |
| `check_single_skill` | Conjunctions in skill name | Post-check — flags only |
| `check_vocabulary_ceiling` | Prompt over word limit | Post-check — flags only |
| `check_cognitive_load` | K-2 hook over 80 words | Post-check — flags only |
| `check_cultural_bias` | Culture-specific terms | Post-check — flags only |

**Key design principle:**
Pre-checks are strict -- they stop bad inputs before wasting an API call.
Post-checks are permissive -- they flag issues but let the lesson through,
because a flagged lesson is more useful than no lesson at all.

**Known limitation:**
The cultural bias check uses a keyword list -- it will miss subtle bias
and may produce false positives. A production system would use a
fine-tuned classifier trained on diverse educational content.
